# 03 - Daily Admin Monitoring archive

This is the notebook to schedule daily. It runs the shared extractor in `daily` mode and then reconciles that exact run. A failure in either child notebook fails this notebook and the schedule.


In [ ]:
extraction_timeout_seconds = 7200
reconciliation_timeout_seconds = 3600


In [ ]:
import json

extract_result_text = notebookutils.notebook.run(
    "02_Extract_Admin_Monitoring_Models",
    extraction_timeout_seconds,
    {"run_mode": "daily"},
)
extract_result = json.loads(extract_result_text)
if extract_result.get("failed") != 0:
    raise RuntimeError(f"Daily extraction reported failures: {extract_result_text}")

reconcile_result_text = notebookutils.notebook.run(
    "04_Reconcile_Admin_Monitoring_Archive",
    reconciliation_timeout_seconds,
    {"run_id": extract_result["run_id"]},
)
reconcile_result = json.loads(reconcile_result_text)
if reconcile_result.get("failed_checks") != 0:
    raise RuntimeError(f"Daily reconciliation reported failures: {reconcile_result_text}")

summary = {
    "run_id": extract_result["run_id"],
    "extraction": extract_result,
    "reconciliation": reconcile_result,
}
print("STEP 03 PASSED")
print(json.dumps(summary, indent=2))
notebookutils.notebook.exit(json.dumps(summary, separators=(",", ":")))
